# 🔌 Notebook 2: Circuit Breaker

When a downstream is *clearly* dead, retries waste resources and make recovery slower. A **circuit breaker** trips after N consecutive failures and rejects calls *fast* until a cooldown elapses.

States: **CLOSED** (normal) → **OPEN** (fast-fail) → **HALF_OPEN** (probe).


## 🛠️ Setup

```bash
cd 04-patterns/resilience
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 Implementation

In [ ]:
import time

class CircuitBreaker:
    def __init__(self, fail_threshold=3, cooldown=2.0):
        self.fail_threshold = fail_threshold
        self.cooldown = cooldown
        self.fails = 0
        self.opened_at = 0.0
        self.state = 'CLOSED'

    def call(self, fn):
        if self.state == 'OPEN':
            if time.monotonic() - self.opened_at >= self.cooldown:
                self.state = 'HALF_OPEN'
                print('  ↪ probing (HALF_OPEN)')
            else:
                raise RuntimeError('circuit OPEN — fast-failing')
        try:
            result = fn()
        except Exception:
            self.fails += 1
            if self.fails >= self.fail_threshold or self.state == 'HALF_OPEN':
                self.state = 'OPEN'
                self.opened_at = time.monotonic()
                print('  ⚡ circuit OPENED')
            raise
        # success
        self.state = 'CLOSED'; self.fails = 0
        return result


## 💥 Watch it trip

In [ ]:
broken = True
def downstream():
    if broken: raise IOError('500')
    return 'ok'

cb = CircuitBreaker(fail_threshold=3, cooldown=1.0)
for i in range(8):
    try:
        print(i, cb.call(downstream))
    except Exception as e:
        print(i, 'FAIL:', e)
    time.sleep(0.3)

print('--- downstream recovers ---')
broken = False
time.sleep(1.1)  # wait out the cooldown
for i in range(3):
    try: print('recovery', cb.call(downstream))
    except Exception as e: print('recovery FAIL:', e)


Without the breaker, every one of those calls would have hung waiting for the slow timeout. With the breaker, we *fast-fail* and protect both the caller and the downstream.

Often combined with **fallbacks**: 'circuit is open → return cached / default value' instead of raising.